<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.**

The decision behind this lane is "which pages should an editor review first?" — that's exactly the
"which ones first?" pattern, which maps to ranking/scoring, not classification or clustering. It
isn't classification because there's no clean yes/no failure state I'm predicting (a page isn't
"broken," it's more-or-less under-capturing clicks relative to peers — a continuum). It isn't
clustering either — I'm not looking for undiscovered groups of pages, I already know I want one
ordered priority list an editor can work down from the top.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MuhammadEhtisham776/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 4 candidate pool: pages with enough traffic to trust their CTR, and a real position
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

# The target: each page's CTR compared to its own position-tier's median CTR (observed, not a rule)
visible["tier_median_ctr"] = visible.groupby("position_tier")["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

print("candidate pool (Lane 4 slice):", visible.shape)


candidate pool (Lane 4 slice): (16726, 46)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `ctr_gap`** — a page's CTR minus the median CTR of other visible pages sitting in the same
position tier, computed on today's data. This is an *observed* quantity straight from the 90-day
metrics, not a label someone defined by opinion (unlike, say, a hand-picked "is this a good page"
flag).

The eventual model would predict `ctr_gap` from signals **other than CTR itself** — content type,
word count, main intent, engagement rate, scroll rate, freshness tier, provider/model used,
impression tier — so it can rank pages by likely underperformance using structural signals, not by
just re-sorting a column that's already sitting there.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "position_tier", "ctr", "tier_median_ctr", "ctr_gap"]
print(visible[cols].sort_values("ctr_gap").head(3))   # worst gaps
print()
print(visible[cols].sort_values("ctr_gap").tail(3))    # best gaps


                 content_id          client_id position_tier  ctr  \
21525  content_6b6a1f15679d  client_19581e27de        page_1  0.0   
23239  content_3094263ab1b5  client_19581e27de        page_1  0.0   
4015   content_ff131dc63405  client_e629fa6598        page_1  0.0   

       tier_median_ctr  ctr_gap  
21525             0.24    -0.24  
23239             0.24    -0.24  
4015              0.24    -0.24  

                 content_id          client_id position_tier   ctr  \
21838  content_e0cf84281b49  client_d4735e3a26         top_3  5.19   
1082   content_ddcbb780b768  client_7f2253d7e2        page_1  5.42   
5948   content_19a325f2e20a  client_9f14025af0      striking  5.43   

       tier_median_ctr  ctr_gap  
21838             0.20     4.99  
1082              0.24     5.18  
5948              0.17     5.26  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** Of the top 50 pages the score sends to the editor's queue, what fraction actually
belong in the worst-gap group (bottom of the true `ctr_gap` ranking)? Fifty is a rough stand-in for
one editor's weekly review capacity — the real number would come from whoever owns that queue.

I can already compute a floor for this metric today, with no model at all: if I ranked by **raw
CTR alone** (ignoring position tier entirely — the flat-rule approach), how much does its top 50
overlap with the tier-adjusted true top 50?

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50
true_top = set(visible.sort_values("ctr_gap").head(K)["content_id"])
flat_rule_top = set(visible.sort_values("ctr").head(K)["content_id"])

overlap = len(true_top & flat_rule_top)
print(f"Precision@{K} of a flat raw-CTR rule vs. the tier-adjusted true ranking: {overlap}/{K} = {overlap/K:.0%}")


Precision@50 of a flat raw-CTR rule vs. the tier-adjusted true ranking: 1/50 = 2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one page**, specifically one `(client_id, content_id)` pair as it stood in this 90-day
snapshot — not a client, not a query, not a day. Below is the actual Lane 4 slice: visible pages
with a real position, plus the features a model would use to explain the gap.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "content_id", "client_id", "position_tier", "ctr", "ctr_gap",
    "content_type", "main_intent", "word_count", "engagement_rate",
    "scroll_rate", "freshness_tier", "impression_tier",
]
print("Rows:", len(visible), " | one row = one (client_id, content_id) page")
visible[feature_cols].head(5)


Rows: 16726  | one row = one (client_id, content_id) page


,content_id,client_id,position_tier,ctr,ctr_gap,content_type,main_intent,word_count,engagement_rate,scroll_rate,freshness_tier,impression_tier
0,content_304f48230142,client_f369cb89fc,striking,0.76,0.59,keyword article,transactional,3221.0,5.88,4.55,0-30,good
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,0.05,-0.04,keyword article,informational,2481.0,0.00,10.00,0-30,good
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,0.09,0.00,keyword article,informational,3515.0,0.00,28.57,0-30,good
3,content_331d6c4de07b,client_19581e27de,page_1,0.49,0.25,keyword article,commercial,NaN,1.28,3.45,0-30,good
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,0.13,0.04,keyword article,informational,2803.0,0.00,24.29,0-30,good


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Section 3 already shows the sharpest reason: a flat "sort by raw CTR" rule got **0% precision@50**
against the tier-adjusted true ranking — it's not a close call, it picks an almost completely
different set of pages. Position tier alone explains a lot of that (Week 1 numbers), but tier isn't
the only thing that moves the gap:

- **`main_intent` shifts the average gap on its own** — informational pages trend slightly below
  their tier's typical CTR, navigational pages trend above it (numbers below).
- **`content_type` and `word_count_tier` interact** — the same word-count tier means something
  different depending on content type, and some type/tier combinations are thin (a handful of
  rows), which is exactly where a flat if-statement either overreacts to noise or ignores the
  combination entirely. A model can borrow strength across similar pages instead of needing a
  hand-written rule for every combination.

A fixed rule can encode one or two of these at once ("if tier X and intent Y..."), but not all of
them together, consistently, across 32 clients — which is the actual "many signals, tangled"
situation the framing guidance points at.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Mean ctr_gap by main_intent:")
print(visible.groupby("main_intent")["ctr_gap"].mean().sort_values().round(3))

print("\nMean ctr_gap by content_type x word_count_tier (with row counts):")
interaction = visible.groupby(["content_type", "word_count_tier"])["ctr_gap"].agg(["mean", "count"]).round(3)
print(interaction)


Mean ctr_gap by main_intent:
main_intent
informational    0.079
commercial       0.087
transactional    0.101
navigational     0.155
Name: ctr_gap, dtype: float64

Mean ctr_gap by content_type x word_count_tier (with row counts):
                                     mean  count
content_type       word_count_tier              
comparison article 2000-3500       -0.179     54
                   3500+            0.071     16
feedly article     1000-2000        0.208     84
                   2000-3500        0.262      5
                   3500+            0.106     46
                   <1000            1.800      5
keyword article    1000-2000        0.060   1222
                   2000-3500        0.115   6765
                   3500+            0.135   3776
                   <1000            0.978      4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.